# Neural Posterior Estimation for LISA
## Part 1 — discrete normalizing flows for galactic binaries

**AI for LISA Hackathon @ ESTEC — tutorial by Stephen Green (Tuesday)**

**Neural posterior estimation** fits a conditional density $q_\phi(\theta \mid d)$ to
prior–simulator pairs by minimizing
$$
\mathcal{L}(\phi) \;=\; \mathbb{E}_{\theta \sim \pi(\theta),\; d \sim p(d \mid \theta)}
\bigl[-\log q_\phi(\theta \mid d)\bigr].
$$
The expectation is over the *joint* distribution $p(\theta,d)$, so up to a $\phi$-independent constant this equals the KL divergence $\mathbb{E}_{p(d)}\bigl[D_{\rm KL}\bigl(p(\theta\mid d)\,\|\,q_\phi(\theta\mid d)\bigr)\bigr]$:
minimized over all densities, the answer is the true posterior.

The negative-log-likelihood (NLL) loss contains $\log q_\phi$, so we must be able to **evaluate**
the density, not just sample it; and $q_\phi$ must handle a correlated, possibly multimodal
posterior. **Discrete normalizing flows** do both. Flow matching (Monday) bypasses the NLL loss and allows training a continuous normalizing flow without density evaluations.

Fast density evaluation is also useful for **importance sampling**, as we see below.

<!-- ---

### Roadmap (about 50 minutes)

| | | |
|---|---|---|
| 1 | The signal model | ~5 min |
| 2 | Setting up NPE | ~5 min |
| 3 | Building a discrete flow | ~10 min |
| 4 | Training | ~8 min |
| 5 | The posterior | ~5 min |
| 6 | Is the posterior right? | ~10 min |
| 7 | Watching the flow transport probability | ~3 min |
| 8 | Exercises | at home / if time remains |

Section 6 may run into the Part 2 hour; the material there is what Part 2 builds on. -->

On Colab: **Runtime → Change runtime type → T4 GPU**, then **Run all**. Training is ~5 min on a T4,
~17 min on CPU.

In [ ]:
# Guarded so that this is a no-op where the packages already exist (e.g. a uv-managed venv,
# whose contents `uv sync` reconciles against the lockfile); on a bare Colab runtime it installs.
import importlib.util
import sys

if any(importlib.util.find_spec(pkg) is None for pkg in ("glasflow", "corner")):
    !{sys.executable} -m pip install -q glasflow corner

`glasflow` wraps [`nflows`](https://github.com/bayesiains/nflows), the reference implementation of
the transforms we use. `torch`, `numpy` and `matplotlib` are already on Colab.

In [ ]:
import os
import time

import corner
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import BatchSampler, DataLoader, Dataset, RandomSampler

from glasflow.nflows import distributions, flows, transforms
import glasflow.nflows as nflows
import glasflow.nflows.nn.nets as nflows_nets

# `mps` is deliberately excluded: nflows has known correctness problems on that backend.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
if device == "cpu":
    print("No GPU detected. On Colab: Runtime -> Change runtime type -> T4 GPU.")

torch.manual_seed(0);

---
# 1. Galactic binary signal model

<!-- *(~5 min)* -->

Detached white-dwarf binaries are LISA's most numerous source ($\sim 10^4$ individually
resolvable) and its simplest: essentially monochromatic, with a slow frequency drift.

For this example, take a **post-search** view: the source is already localized in frequency to $\pm 0.1\,\mu$Hz.
Demodulating by a pure carrier $e^{2\pi i f_{\rm ref} t}$ at $f_{\rm ref} = 3$ mHz and keeping the
slow **complex envelope** turns four years of data into $N = 256$ complex samples rather than
$\sim\!5\times10^{8}$ raw ones. The **residual** phase left by the demodulation is
$$
\Phi(t) \;=\; 2\pi\Bigl(\delta f\, t + \tfrac{1}{2}\dot f\, t^2\Bigr) \;+\; \phi_0
\;+\; 2\pi f_{\rm ref}\,\frac{R}{c}\,\cos\beta \,\cos\!\Bigl(\frac{2\pi t}{\rm yr} - \lambda\Bigr),
\qquad R = 1\,{\rm AU},
$$
$$
h(t) \;=\; A\Bigl[F_+(t)\,a_+ \;+\; i\,F_\times(t)\,a_\times\Bigr]\,e^{i\Phi(t)},
\qquad a_+ = \frac{1+\cos^2\iota}{2},\quad a_\times = \cos\iota .
$$

- The carrier is a *pure* tone, so the Doppler term survives: LISA's motion around the Sun,
  modulation index $2\pi f_{\rm ref}(R/c)\cos\beta \approx 8.3$ rad. It is what localizes a source on
  the sky; here $(\beta, \lambda) = (0.5, 1.0)$, fixed and known.
- $F_+, F_\times$ are a **schematic amplitude modulation at two cycles per year, not a TDI
  response**. A real one (`fastGB`) would change none of the machine learning below.
- $d = h + n$, with $n$ white Gaussian noise of unit variance on each of the $2N = 512$ real
  components. White is right here: across the $\approx 2\,\mu$Hz band the envelope spans, the LISA
  PSD varies by only about $0.2\%$, so whitening is a constant rescale and $A$ is an amplitude in
  units of the noise $\sigma$.

Over $T_{\rm obs} = 4$ years,
$$
\theta = (A,\; \delta f,\; \dot f,\; \phi_0,\; \cos\iota), \qquad
A \sim \mathcal{U}[1.4, 7], \quad \delta f \sim \mathcal{U}[-1, 1]\times 10^{-7}\,{\rm Hz},
$$
$$
\dot f \sim \mathcal{U}[3, 10]\times10^{-17}\,{\rm Hz/s}, \quad
\phi_0 \sim \mathcal{U}[0, 2\pi], \quad \cos\iota \sim \mathcal{U}[-1, 1].
$$
For the drift prior $\dot f$, over four years the binary accumulates only $1.5$–$5.0$ radians of extra
phase. So $\dot f$ is measurable, but only *marginally*. You will see that in the posterior.

<!-- The simulator is shared verbatim with **Part 2** on Thursday (Max Dax, flow matching, identical
data). It is deliberately schematic: realistic inference structure at negligible cost, not physical
accuracy. -->

In [ ]:
YEAR = 3.15581498e7          # sidereal-ish year [s]
T_OBS = 4 * YEAR             # observation time (LISA nominal mission)
N = 256                      # samples of the complex envelope
F_REF = 3e-3                 # heterodyne reference frequency [Hz]
R_AU_C = 499.0               # astronomical unit in light seconds
BETA, LAM = 0.5, 1.0         # ecliptic latitude / longitude (fixed & known here)
NOISE_STD = 1.0              # whitened noise, per real sample

t = torch.linspace(0, T_OBS, N, device=device)

PRIOR_LOW = torch.tensor([1.4, -1e-7, 3e-17, 0.0, -1.0], device=device)
PRIOR_HIGH = torch.tensor([7.0, 1e-7, 1e-16, 2 * np.pi, 1.0], device=device)
LABELS = [r"$A$", r"$\delta f\ [10^{-7}\,{\rm Hz}]$", r"$\dot f\ [10^{-17}\,{\rm Hz/s}]$",
          r"$\phi_0$", r"$\cos\iota$"]
PLOT_SCALE = torch.tensor([1.0, 1e7, 1e17, 1.0, 1.0], device=device)  # readable axes


def sample_prior(n):
    """Draw n parameter vectors from the (uniform, box) prior. -> (n, 5)"""
    return PRIOR_LOW + (PRIOR_HIGH - PRIOR_LOW) * torch.rand(n, 5, device=device)


def gb_envelope(theta):
    """Complex envelope of the heterodyned galactic-binary signal. (B, 5) -> (B, N) complex."""
    A, df, fdot, phi0, cosi = (theta[:, i:i + 1] for i in range(5))
    # intrinsic phase evolution relative to F_REF
    phase = 2 * np.pi * (df * t + 0.5 * fdot * t**2) + phi0
    # Doppler modulation from the orbit around the Sun
    phase = phase + 2 * np.pi * F_REF * R_AU_C * np.cos(BETA) * torch.cos(2 * np.pi * t / YEAR - LAM)
    # inclination-dependent polarization amplitudes
    a_plus, a_cross = (1 + cosi**2) / 2, cosi
    # schematic LISA-like antenna modulation (2 cycles/yr), not a TDI response
    f_plus = 0.5 * (1 + 0.6 * torch.cos(4 * np.pi * t / YEAR - 2 * LAM))
    f_cross = 0.5 * (1 + 0.6 * torch.sin(4 * np.pi * t / YEAR - 2 * LAM))
    return A * (f_plus * a_plus + 1j * f_cross * a_cross) * torch.exp(1j * phase)


def signal(theta):
    """Noise-free network features. (B, 5) -> (B, 2N) real, [Re, Im] concatenated."""
    y = gb_envelope(theta)
    return torch.cat([y.real, y.imag], dim=-1)


def simulate(theta):
    """(B, 5) -> (B, 2N) real network features [Re, Im], with whitened noise."""
    d = signal(theta)
    return d + NOISE_STD * torch.randn_like(d)


def log_likelihood(theta, d):
    """Exact Gaussian log likelihood (up to a constant), used for validation in Part 2."""
    return -0.5 * ((d - signal(theta)) ** 2).sum(-1) / NOISE_STD**2

Five short vectorized functions, so a batch of signals costs essentially nothing. Sanity check: with
unit-variance noise per real component, the optimal SNR is just $\rho = \lVert h \rVert$.

In [ ]:
snr = signal(sample_prior(2000)).pow(2).sum(-1).sqrt()
print(f"prior SNR range: {snr.min():.0f} .. {snr.max():.0f}   (median {snr.median():.0f})")

A median SNR near 30 is comfortable: the posterior is much narrower than the prior.

The envelope is sampled every 5.7 days — a Nyquist band of $\pm 1.0\,\mu$Hz, comfortably above
everything in it: the beat against $f_{\rm ref}$ ($\le 0.1\,\mu$Hz), the annual Doppler swing
($0.26\,\mu$Hz), the antenna modulation ($0.06\,\mu$Hz).

In [ ]:
# One source seen two ways: the whole record, and two hours of it.
theta_show = torch.tensor([[5.0, 8e-8, 6e-17, 1.0, 0.15]], device=device)
env = gb_envelope(theta_show)[0].cpu().numpy()      # the 256 complex envelope samples
amp = np.abs(env)
t_s = t.cpu().numpy()

N_CYC = 22                                          # carrier cycles to resolve in the zoom
k = int(amp.argmax())                               # window at the antenna-pattern maximum
t0 = t_s[k]
t_fine = t0 + np.linspace(0, N_CYC / F_REF, 2000)   # ~2 h, over which the envelope is flat
carrier = (env[k] * np.exp(2j * np.pi * F_REF * t_fine)).real

n_carrier = F_REF * T_OBS                           # carrier cycles in the record
n_env = float(PRIOR_HIGH[1].item()) * T_OBS         # envelope cycles, widest allowed by the prior

fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
ax = axes[0]
# The carrier cannot be resolved at this scale, so draw the band it fills.
ax.fill_between(t_s / YEAR, -amp, amp, color="C0", alpha=0.3, lw=0)
ax.plot(t_s / YEAR, amp, color="C0", lw=1.2)
ax.plot(t_s / YEAR, -amp, color="C0", lw=1.2)
ax.axvline(t0 / YEAR, color="C3", lw=1.2)
ax.annotate("2 h window", xy=(t0 / YEAR, 1.02 * amp.max()), xytext=(t0 / YEAR + 0.45, 1.2 * amp.max()),
            color="C3", fontsize=9, arrowprops=dict(arrowstyle="->", color="C3", lw=1.0))
ax.set_xlim(0, T_OBS / YEAR); ax.set_ylim(-1.35 * amp.max(), 1.35 * amp.max())
ax.set_xlabel("$t$ [yr]"); ax.set_ylabel("$h(t)$")
ax.set_title(f"the whole record: {n_carrier:,.0f} carrier cycles", fontsize=10)

ax = axes[1]
ax.plot((t_fine - t0) / 3600, carrier, color="C0", lw=1.0)
ax.axhline(amp[k], color="C0", lw=1.2, ls="--", alpha=0.7)     # the same band, close up
ax.axhline(-amp[k], color="C0", lw=1.2, ls="--", alpha=0.7)
ax.set_xlabel("$t - t_0$ [h]")
ax.set_title(f"two hours of it: {N_CYC} cycles, resolved", fontsize=10)
plt.tight_layout()

print(f"carrier cycles over 4 years:              {n_carrier:,.0f}")
print(f"envelope cycles, widest allowed by prior: {n_env:.1f}")

Demodulating by the known $f_{\rm ref}$ removes about 380,000 cycles and leaves at most thirteen, the
ones the source parameters modulate. Four years of data are reduced to just 256 complex samples, as we see below.

In [ ]:
torch.manual_seed(1)
theta_demo = sample_prior(2)
clean, noisy = signal(theta_demo), simulate(theta_demo)
t_yr = (t / YEAR).cpu()

dt = (t[1] - t[0]).item()                                     # 5.7 d between envelope samples
freq = np.fft.fftshift(np.fft.fftfreq(4 * N, d=dt)) * 1e6     # offset from F_REF, in uHz
window = np.hanning(N)                                        # keeps leakage off the comb

fig, axes = plt.subplots(3, 2, figsize=(11, 6.6))
for j in range(2):
    rho = clean[j].pow(2).sum().sqrt()
    for k, (part, sl) in enumerate([("Re", slice(0, N)), ("Im", slice(N, 2 * N))]):
        ax = axes[k, j]
        ax.plot(t_yr, noisy[j, sl].cpu(), color="0.75", lw=0.8, label="data $d$")
        ax.plot(t_yr, clean[j, sl].cpu(), color=f"C{j}", lw=1.5, label="signal $h$")
        ax.set_ylabel(rf"{part}$\,h$")
    axes[0, j].set_title(rf"$A$={theta_demo[j, 0]:.1f}, $\cos\iota$={theta_demo[j, 4]:+.2f},"
                         rf"  SNR={rho:.0f}", fontsize=10)
    axes[1, j].set_xlabel("$t$ [yr]")

    # amplitude spectrum of the clean complex envelope: a comb of 1/yr sidebands, not a spike
    y = (clean[j, :N] + 1j * clean[j, N:]).cpu().numpy()
    amp = np.abs(np.fft.fftshift(np.fft.fft(y * window, 4 * N))) / window.sum()
    ax = axes[2, j]
    ax.plot(freq, amp, color=f"C{j}", lw=0.9)
    ax.set_yscale("log"); ax.set_ylim(3e-3 * amp.max(), 3 * amp.max()); ax.set_xlim(-0.5, 0.5)
    ax.set_xlabel(r"$f - f_{\rm ref}$ [$\mu$Hz]"); ax.set_ylabel(r"$|\tilde h|$")
axes[0, 0].legend(fontsize=8, loc="upper right")
plt.tight_layout()

The information is in the slow beating of $\delta f$ and $\dot f$ against $f_{\rm ref}$, the annual
antenna envelope, and the Re/Im ratio set by $\cos\iota$ — in the frequency domain, a comb of
sidebands spaced $1/{\rm yr}$ and spanning $\pm0.3\,\mu$Hz (Doppler modulation index $\approx 8$ rad)
rather than a single spike. The FFT is a unitary change of basis and the noise is white, so the 512
numbers say the same thing either way; we stay in the time domain, where the modulation is legible.

---
# 2. Data preparation

<!-- *(~5 min)* -->

Draw $\theta \sim \pi(\theta)$, simulate $d \sim p(d \mid \theta)$, minimize
$\mathbb{E}\bigl[-\log q_\phi(\theta \mid d)\bigr]$. Two practical points first.

- **Standardization.** The parameters span seventeen orders of magnitude ($\dot f \sim 10^{-17}$,
  $\phi_0 \sim 1$), and networks want $O(1)$ inputs and outputs. Rescale by the prior mean and
  standard deviation — $(\text{range})/\sqrt{12}$ for a uniform prior — and undo it after sampling.
  The data is already whitened.
- **Noise on the fly.** Generate noise-free signals once; add a *fresh* noise realization every time
  one is used. The waveform is the expensive object and a Gaussian draw is free, and it is free
  augmentation — the network never sees the same example twice. Exercise C turns it off.

In [ ]:
N_TRAIN = 100_000
N_VAL = 10_000
BATCH_SIZE = 512

THETA_MEAN = (PRIOR_LOW + PRIOR_HIGH) / 2
THETA_STD = (PRIOR_HIGH - PRIOR_LOW) / np.sqrt(12)   # sd of a uniform distribution


def make_signal_set(n, chunk=10_000):
    """Prior draws and their noise-free signals. Parameters come back standardized.

    Parameters
    ----------
    n : int
        Number of examples.
    chunk : int, optional
        Number of waveforms simulated at a time, to bound peak memory.

    Returns
    -------
    theta : torch.Tensor
        Standardized parameters, shape (n, 5).
    signals : torch.Tensor
        Noise-free features, shape (n, 2N).
    """
    theta = sample_prior(n)
    s = torch.cat([signal(theta[i:i + chunk]) for i in range(0, n, chunk)])
    return (theta - THETA_MEAN) / THETA_STD, s


t0 = time.time()
theta_train, signals_train = make_signal_set(N_TRAIN)
theta_val, signals_val = make_signal_set(N_VAL)
mb = signals_train.element_size() * signals_train.nelement() / 1e6
print(f"{N_TRAIN:,} training signals of shape {tuple(signals_train.shape[1:])} "
      f"({mb:.0f} MB) in {time.time() - t0:.1f} s")

In PyTorch, we prepare data as a subclass of the `Dataset` class. We have to define a `__getitem__` method, which in our case returns a $(d, \theta)$ pair. Note that it draws the noise *at access time*, so the same index
gives different data each time. This is how we add random noise realizations on the fly.

It also takes a whole batch of indices at once (a `BatchSampler`, with `batch_size=None`), which
draws the batch's noise in one vectorized call instead of collating 512 separate tensors — otherwise
that dominates step time on Colab's host CPU.

In [ ]:
class GBDataset(Dataset):
    """Noise-free signals and parameters; noise is drawn afresh on every access."""

    def __init__(self, theta, signals):
        self.theta = theta
        self.signals = signals

    def __len__(self):
        return len(self.theta)

    def __getitem__(self, idx):
        s = self.signals[idx]
        return s + NOISE_STD * torch.randn_like(s), self.theta[idx]


def make_loader(dataset, batch_size=BATCH_SIZE, shuffle=True):
    """DataLoader that passes a whole batch of indices to __getitem__ at once.

    Parameters
    ----------
    dataset : Dataset
        Dataset whose __getitem__ accepts a list of indices.
    batch_size : int, optional
        Examples per batch.
    shuffle : bool, optional
        Shuffle and drop the last partial batch (training); False keeps the fixed
        order and every example (validation).

    Returns
    -------
    DataLoader
        Yields (data, parameters) batches.
    """
    order = RandomSampler(dataset) if shuffle else range(len(dataset))
    sampler = BatchSampler(order, batch_size=batch_size, drop_last=shuffle)
    return DataLoader(dataset, sampler=sampler, batch_size=None)


train_dataset = GBDataset(theta_train, signals_train)
val_dataset = GBDataset(theta_val, signals_val)
train_loader = make_loader(train_dataset)
val_loader = make_loader(val_dataset, shuffle=False)

d_batch, theta_batch = next(iter(train_loader))
print(f"batch: data {tuple(d_batch.shape)}, parameters {tuple(theta_batch.shape)}")

In [ ]:
# The same index, accessed twice, gives two different noise realizations.
d1, _ = train_dataset[[0]]
d2, _ = train_dataset[[0]]

plt.figure(figsize=(9, 2.6))
plt.plot(d1[0, :N].cpu(), color="C0", lw=0.7, alpha=0.8, label="access 1")
plt.plot(d2[0, :N].cpu(), color="C1", lw=0.7, alpha=0.8, label="access 2")
plt.plot(signals_train[0, :N].cpu(), "k", lw=1.6, label="underlying signal")
plt.xlabel("sample"); plt.ylabel(r"Re$\,d$"); plt.legend(fontsize=8, ncol=3)
plt.tight_layout()

---
# 3. Building a discrete normalizing flow

<!-- *(~10 min)* -->

Fix the data $d$. A flow represents $q(\theta \mid d)$ as the pushforward of a **base density**
$\pi(u) = \mathcal{N}(0, I)$ through a learned invertible map. Let
$f_d : \mathbb{R}^D \to \mathbb{R}^D$ run in the *normalizing* direction, $u = f_d(\theta)$; change
of variables gives
$$
\boxed{\;\log q(\theta \mid d) \;=\; \log \pi\bigl(f_d(\theta)\bigr)
\;+\; \log\bigl|\det J_{f_d}(\theta)\bigr|\;}
\qquad J_{f_d} = \frac{\partial u}{\partial \theta}.
$$
To *sample*, run it backwards: draw $u \sim \mathcal{N}(0, I)$, set $\theta = f_d^{-1}(u)$.
Conditioning $f_d$ on $d$ is what makes the inference amortized.

### Why the architecture is constrained

- **$f_d$ must be invertible**, or the formula above defines nothing. That alone rules out a
  generic MLP.
- **$\det J_{f_d}$ must be cheap.** A general determinant is $O(D^3)$, and assembling the Jacobian of
  a generic network takes $D$ backward passes *per training example* — affordable at $D = 5$, not at
  $D = 15$ for a compact binary, still less for a global fit.
- Flow matching removes exactly this constraint; it is what discrete flows pay for an exact density
  in one pass.
- The resolution: compose **simple invertible maps**, each with a triangular Jacobian. Inverses
  compose in reverse order and log-determinants add, so composition is harmless.

### The autoregressive step

Impose an ordering on the parameters and let each output depend only on the inputs that precede it:
$$
u_i \;=\; \frac{\theta_i - \mu_i(\theta_{1:i-1},\, d)}{\sigma_i(\theta_{1:i-1},\, d)} .
$$
Then $\partial u_i / \partial \theta_j = 0$ for $j > i$, the Jacobian is triangular, and
$$
\log|\det J| \;=\; -\sum_{i=1}^{D} \log \sigma_i .
$$
The functions $\mu_i, \sigma_i$ may be arbitrarily complicated neural networks — all of the
expressive power lives there, and none of it costs anything in the determinant. This is the key
trick.

Equivalently, the step is an autoregressive factorization
$q(\theta \mid d) = \prod_i q(\theta_i \mid \theta_{1:i-1}, d)$ in which each conditional is Gaussian
in $\theta_i$. Note that the *joint* is not Gaussian even so: a chain of nonlinear conditional
Gaussians already produces curved, strongly non-Gaussian distributions.

A **MADE** network computes every $\mu_i, \sigma_i$ in a single masked forward pass, so evaluating
the density — the direction training needs — costs one network pass. Sampling has to run one
component at a time ($\theta_1$ from $u_1$, then $\theta_2$ from $u_2$ and $\theta_1$, and so on), so
it costs $D$ passes. At $D = 5$ that is irrelevant. It is, however, the reason production models
prefer **coupling** transforms, which are fast in both directions; Exercise A tries one.

### Stacking, and conditioning

- **Why stack.** One affine step is limited: the first conditional is a plain Gaussian and the
  ordering is arbitrary. Between steps we put a **random permutation** ($\log|\det J| = 0$) and a
  **learned LU-parametrized linear map** ($\log|\det J| = \sum_i \log|U_{ii}|$, inverse = two
  triangular solves). The linear layers rotate into the directions the posterior is actually
  correlated along — most of the work in this problem.
- **Conditioning.** Every $\mu, \sigma$ takes $d$ as an extra input, and we feed the raw 512 features
  straight in, **with no embedding network**. Production DINGO would put an SVD-initialized linear
  compression and a large ResNet in between; here the flow's own input layers cope. Part 2 picks this
  up.

Code below adapted from the [neural spline flow repository](https://github.com/bayesiains/nsf).

In [ ]:
def create_base_transform(i, param_dim, context_dim, hidden_dim=128,
                          num_transform_blocks=2, base_transform_type="maf"):
    """One expressive flow step: the part that carries the learned nonlinearity.

    Parameters
    ----------
    i : int
        Index of the flow step; only used to alternate the coupling mask.
    param_dim : int
        Dimension D of theta.
    context_dim : int
        Dimension of the conditioning data d.
    hidden_dim : int, optional
        Hidden width of the MADE or ResidualNet.
    num_transform_blocks : int, optional
        Number of residual blocks inside that network.
    base_transform_type : {"maf", "rq-coupling"}, optional
        Masked affine autoregressive, or rational-quadratic spline coupling.

    Returns
    -------
    transforms.Transform
        The conditional transform for one flow step.
    """
    activation = nn.ELU()

    if base_transform_type == "maf":
        # Masked autoregressive: one MADE pass gives all (mu_i, sigma_i).
        return transforms.MaskedAffineAutoregressiveTransform(
            param_dim,
            hidden_features=hidden_dim,
            context_features=context_dim,
            num_blocks=num_transform_blocks,
            activation=activation,
            use_batch_norm=False,
        )

    elif base_transform_type == "rq-coupling":
        # Rational-quadratic spline coupling: half the parameters are passed through
        # unchanged and used to predict a monotonic spline for the other half.
        mask = nflows.utils.create_alternating_binary_mask(param_dim, even=(i % 2 == 0))
        return transforms.PiecewiseRationalQuadraticCouplingTransform(
            mask=mask,
            transform_net_create_fn=lambda in_features, out_features: nflows_nets.ResidualNet(
                in_features=in_features,
                out_features=out_features,
                hidden_features=hidden_dim,
                context_features=context_dim,
                num_blocks=num_transform_blocks,
                activation=activation,
                use_batch_norm=False,
            ),
            num_bins=8,
            tails="linear",
            tail_bound=5.0,
            apply_unconditional_transform=False,
        )

    raise ValueError(f"unknown base_transform_type: {base_transform_type}")


def create_linear_transform(param_dim):
    """Permutation (log|det J| = 0) followed by a learned LU-parametrized linear map."""
    return transforms.CompositeTransform([
        transforms.RandomPermutation(features=param_dim),
        transforms.LULinear(param_dim, identity_init=True),
    ])


def create_flow(num_flow_steps, param_dim, context_dim, **base_transform_kwargs):
    """A standard normal base distribution followed by num_flow_steps (linear + base) steps.

    Parameters
    ----------
    num_flow_steps : int
        Number of (linear + base) steps; one further linear map is appended.
    param_dim : int
        Dimension D of theta.
    context_dim : int
        Dimension of the conditioning data d.
    **base_transform_kwargs
        Forwarded to `create_base_transform` (hidden_dim, base_transform_type, ...).

    Returns
    -------
    flows.Flow
        Conditional flow supporting `.log_prob(theta, d)` and `.sample(n, d)`.
    """
    transform = transforms.CompositeTransform(
        [
            transforms.CompositeTransform([
                create_linear_transform(param_dim),
                create_base_transform(i, param_dim, context_dim, **base_transform_kwargs),
            ])
            for i in range(num_flow_steps)
        ]
        + [create_linear_transform(param_dim)]
    )
    base_distribution = distributions.StandardNormal(shape=[param_dim])
    return flows.Flow(transform, base_distribution)

In [ ]:
PARAM_DIM = 5           # (A, df, fdot, phi0, cos iota)
CONTEXT_DIM = 2 * N     # 512 raw data features, no embedding network

flow = create_flow(
    num_flow_steps=5,
    param_dim=PARAM_DIM,
    context_dim=CONTEXT_DIM,
    hidden_dim=128,
    base_transform_type="maf",
).to(device)

n_params = sum(p.numel() for p in flow.parameters())
print(f"{n_params:,} trainable parameters\n")

# In the order density evaluation applies them.
blocks = list(flow._transform._transforms)
for i, block in enumerate(blocks):
    parts = list(block._transforms)
    last = i == len(blocks) - 1
    if not last:
        parts = list(parts[0]._transforms) + parts[1:]      # unpack the step's linear map
    label = "final linear map" if last else f"flow step {i + 1}"
    print(f"  {label:<17} " + " -> ".join(type(p).__name__ for p in parts))

### Reading that structure

Density evaluation ($\theta \to u$, the direction training uses) walks down the list. Per flow step:

- `RandomPermutation` reshuffles the components; $\log|\det J| = 0$.
- `LULinear` applies a learned $5\times5$ matrix $A = LU$; $\log|\det J| = \sum_i \log|U_{ii}|$.
- `MaskedAffineAutoregressiveTransform` runs one MADE pass on $(\theta, d)$, emitting all ten numbers
  $\mu_1, \log\sigma_1, \ldots$ at once; masking guarantees $\mu_i, \sigma_i$ depend only on
  $\theta_{1:i-1}$ and $d$. Then $\theta_i \mapsto (\theta_i - \mu_i)/\sigma_i$,
  $\log|\det J| = -\sum_i \log\sigma_i$.

Nothing ever *inverts* the MADE — that is needed only for sampling. The loss is the minibatch mean of
$-\log q$: one forward pass, one number, no ODE and no sampling inside it.

---
# 4. Training

<!-- *(~8 min)* -->

We use the Adam optimizer with a cosine-annealed learning rate. Reducing the learning rate over time (following a cosine trajectory) is standard, and enables the network to reach a lower final loss value. Caveat: with the learning rate dropping to zero, it will always look like the network has converged, so it is important to have a good idea of how long training will take in the first place. If uncertain, try training first with a constant, or reduce-on-plateau learning rate scheduler.

Note: A deep affine flow can produce an enormous loss in its first few steps, and the gradients sometimes overflow.
`clip_grad_norm_` propagates non-finite values rather than removing them, so we use the norm it
returns as a health check and skip the update when it is not finite — one `nan` poisons every weight
permanently.

In [ ]:
def train_epoch(flow, loader, optimizer, max_grad_norm=5.0):
    """One pass over the loader. Returns (mean loss, number of skipped updates)."""
    flow.train()
    total, n, skipped = 0.0, 0, 0
    for d, theta in loader:
        loss = -flow.log_prob(theta, d).mean()          # E[-log q(theta | d)]
        optimizer.zero_grad()
        loss.backward()
        # clip_grad_norm_ returns the norm *before* clipping, and happily propagates
        # inf/nan. Use it as a health check and drop the update if it is not finite.
        grad_norm = torch.nn.utils.clip_grad_norm_(flow.parameters(), max_grad_norm)
        if torch.isfinite(grad_norm):
            optimizer.step()
            total += loss.item() * len(d)
            n += len(d)
        else:
            skipped += 1
    return total / max(n, 1), skipped


# Orthogonal to eval(): eval() leaves training mode, no_grad() stops autograd recording, so this
# costs less memory and time.
@torch.no_grad()
def evaluate(flow, loader):
    """Mean -log q(theta | d) per example over the loader."""
    flow.eval()
    total, n = 0.0, 0
    for d, theta in loader:
        total += -flow.log_prob(theta, d).sum().item()
        n += len(d)
    return total / n


def train(flow, train_loader, val_loader, epochs, lr=1e-3, report_every=8):
    """Fit the flow by minibatch NLL. report_every=0 prints only the final epoch.

    Parameters
    ----------
    flow : flows.Flow
        The conditional flow to fit, updated in place.
    train_loader, val_loader : DataLoader
        Training and validation batches.
    epochs : int
        Passes over the training set; also the length of the cosine schedule.
    lr : float, optional
        Initial Adam learning rate.
    report_every : int, optional
        Print every this many epochs; 0 prints only the last.

    Returns
    -------
    dict
        {"train": [...], "val": [...]}, one loss per epoch.
    """
    optimizer = torch.optim.Adam(flow.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train": [], "val": []}
    total_skipped = 0
    t0 = time.time()
    for epoch in range(epochs):
        loss, skipped = train_epoch(flow, train_loader, optimizer)
        total_skipped += skipped
        history["train"].append(loss)
        history["val"].append(evaluate(flow, val_loader))
        scheduler.step()
        if epoch == epochs - 1 or (report_every and epoch % report_every == 0):
            note = f"   [{total_skipped} updates skipped so far]" if total_skipped else ""
            print(f"epoch {epoch:4d}   train {history['train'][-1]:8.3f}"
                  f"   val {history['val'][-1]:8.3f}   ({time.time() - t0:.0f} s){note}")
    return history

Training runs here by default (`TRAIN_LIVE = True`), about 5 minutes on a T4 and 17 on a laptop
CPU. Set it `False` to load the checkpoint shipped with the repository instead — which is what
produced the stored outputs below.

In [ ]:
TRAIN_LIVE = True                          # False -> load the shipped checkpoint instead
CHECKPOINT = "checkpoints/part1_flow_v2.pt"
N_EPOCHS = 128                             # ~5 min on a T4, ~17 min on a laptop CPU

if TRAIN_LIVE:
    history = train(flow, train_loader, val_loader, N_EPOCHS)
    os.makedirs("checkpoints", exist_ok=True)
    torch.save({"state_dict": flow.state_dict(), "history": history,
                "n_epochs": N_EPOCHS, "device": device}, CHECKPOINT)
    print(f"saved {CHECKPOINT}")
else:
    # If you are not running from a clone of the repository, fetch the checkpoint first:
    # !mkdir -p checkpoints && wget -q -O checkpoints/part1_flow_v2.pt \
    #     https://github.com/stephengreen/LISAAI-Hackathon-ESTEC/raw/main/checkpoints/part1_flow_v2.pt
    ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
    flow.load_state_dict(ckpt["state_dict"])
    history = ckpt["history"]
    print(f"loaded {CHECKPOINT} (trained for {ckpt['n_epochs']} epochs on {ckpt['device']})")

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(history["train"], label="train")
plt.plot(history["val"], label="validation")
plt.xlabel("epoch"); plt.ylabel(r"$-\log q(\theta \mid d)$")
plt.legend(); plt.tight_layout()
print(f"final: train {history['train'][-1]:.3f}, validation {history['val'][-1]:.3f}")

The loss is a negative log density in standardized parameters, so it can and does go negative: the
posterior is far narrower than the unit-scale base distribution. Only differences between runs carry
information. One nat means $e$ times more density on the true parameters, which is how to compare
architectures (Exercises A and B).

Train and validation track each other closely: with fresh noise on every access the network never
sees the same example twice, so there is very little to overfit to.

---
# 5. The posterior

<!-- *(~5 min)* -->

We fix one observation — the $\theta_{\rm true}$ and noise seed set just below — and sample. Inference is fast, and amortized over choices of events.

In [ ]:
flow.eval()
theta_true = torch.tensor([[4.2, 3.75e-8, 6e-17, 2.0, 0.6]], device=device)
torch.manual_seed(11)                      # fixes the noise realization of this observation
d_obs = simulate(theta_true)

N_POST = 20_000
t0 = time.time()
with torch.no_grad():
    samples = flow.sample(N_POST, d_obs)[0] * THETA_STD + THETA_MEAN
print(f"{N_POST:,} posterior samples in {time.time() - t0:.2f} s")

In [ ]:
fig = corner.corner(
    (samples * PLOT_SCALE).cpu().numpy(),
    labels=LABELS,
    truths=(theta_true * PLOT_SCALE)[0].cpu().numpy(),
    color="C0",
    truth_color="C3",
    levels=(0.5, 0.9),
    hist_kwargs={"density": True},
)
fig.suptitle("NPE posterior (blue) and true parameters (red)", y=1.0);

In [ ]:
# The same 20,000 samples, now drawn on the prior box instead of on themselves.
prior_range = list(zip((PRIOR_LOW * PLOT_SCALE).cpu().numpy(),
                       (PRIOR_HIGH * PLOT_SCALE).cpu().numpy()))
fig = corner.corner(
    (samples * PLOT_SCALE).cpu().numpy(),
    labels=LABELS,
    range=prior_range,
    truths=(theta_true * PLOT_SCALE)[0].cpu().numpy(),
    color="C0",
    truth_color="C3",
    bins=40,
    smooth=1.0,
    levels=(0.5, 0.9),
    fill_contours=True,
    plot_datapoints=False,
    plot_density=False,
    hist_kwargs={"density": True},
)
fig.suptitle("the same posterior, on the prior axes", y=1.0)

# How much of the prior box does it occupy?
s = samples.cpu().numpy()
names = ["A", "df", "fdot", "phi0", "cos i"]
frac = s.std(0) / (PRIOR_HIGH - PRIOR_LOW).cpu().numpy()   # sigma_i / prior width_i
corr = np.corrcoef(s.T)
shrink = np.sqrt(np.linalg.det(corr))                      # correlations shrink the joint volume

print("   ".join(f"{n} {100 * f:.2f}%" for n, f in zip(names, frac)))
print(f"\nproduct of the marginal widths:      {frac.prod():.2e} of the prior box")
print(f"times sqrt(det C) = {shrink:.3f}:            {frac.prod() * shrink:.2e}")
print("\ncorrelation matrix C")
print(np.round(corr, 2))

### Reading the posterior

The truth sits inside every marginal, and the second figure gives the scale of the answer: on the
prior axes the posterior is a needle. The five marginal widths $\sigma_i$, each as a fraction of its
prior range, multiply to $4\times10^{-8}$ of the prior box, and the correlations shrink the joint
volume by a further factor of eleven, to $4\times10^{-9}$.

The widths are very unequal.

- $\delta f$ is by far the best measured: $\sigma \approx 6\times10^{-10}$ Hz, $0.31\%$ of its prior
  range. Placing a ridge that sharp inside a broad prior is the hard part of this problem, and why
  training takes minutes rather than seconds. Part 2a widens the $\delta f$ prior by a factor of 20,
  which shrinks the target by the same factor along that one axis.
- $\dot f$ is the weakest, at $12\%$: $\pm 1\sigma$ covers a quarter of the prior and the marginal is
  visibly broad. The drift is detected, but only just, which is the marginal measurability the prior
  was chosen to represent. Read no more into that panel than it shows.
- $\phi_0$ ($1.7\%$) and $\cos\iota$ ($6.3\%$) sit in between. $A$ is at $10\%$, and only because of
  the degeneracy below.

Two degeneracies dominate the shape, and the printed correlation matrix names them.

- **$A$ and $\cos\iota$**, correlation $-0.97$: both set the observed amplitude, through
  $a_+ = (1+\cos^2\iota)/2$ and $a_\times = \cos\iota$. A more edge-on source is weaker and a larger
  $A$ compensates, the distance–inclination degeneracy familiar from ground-based sources. It breaks
  only weakly, since $a_+$ and $a_\times$ multiply *different* antenna patterns and the annual
  modulation is shallow. What survives is a long, thin, curved banana that runs into
  $\cos\iota = 1$. A flow is supported on all of $\mathbb{R}^5$ and knows nothing of the prior box,
  so it puts a per cent or two of its samples outside; §6.2 gives those zero weight.
- **$\delta f$, $\dot f$, $\phi_0$**, correlations $-0.83$, $-0.69$, $+0.63$: over a finite
  observation a frequency offset, a linear drift and an initial phase generate nearly the same phase
  evolution. That truncated-polynomial near-degeneracy is why $\dot f$ comes out so much broader than
  $\delta f$, and representing the narrow curved ridge is where `LULinear` earns its place.

A plausible corner plot is not evidence that any of this is right. Section 6 checks it.

---
# 6. Validation: Is the posterior right?

<!-- *(~10 min, and it can run into the Part 2 hour)* -->

We perform three checks:

1. compare against an exact sampler on this one observation (§6.1);
2. reweight the flow's own samples by the exact likelihood (§6.2);
3. test coverage over many observations drawn from the prior (§6.3).

Only the first needs a converged chain, which is the cost NPE exists to avoid. The other two are
cheap enough to run on every event.

## 6.1 A reference posterior

The likelihood is available in closed form here, so we can compute the posterior a second way and
compare. Random-walk Metropolis, in the standardized coordinates $u = (\theta - \bar\theta)/\sigma$
the network already uses: 256 chains, three pilot stages that re-estimate the proposal covariance
from the ensemble, then a production stage thinned by 4. The proposal scale $2.38/\sqrt{D}$ is the
standard optimal-scaling choice for a Gaussian target.

The prior enters as a *density*, uniform inside the box and zero outside, not as an additive
constant. We need the same function in §6.2.

In [ ]:
LOG_PRIOR_BOX = -torch.log(PRIOR_HIGH - PRIOR_LOW).sum()


def log_prior(theta):
    """Uniform on the prior box, -inf outside. (B, 5) -> (B,)"""
    inside = ((theta > PRIOR_LOW) & (theta < PRIOR_HIGH)).all(-1)
    return torch.where(inside, LOG_PRIOR_BOX, torch.tensor(-torch.inf, device=theta.device))


def reference_posterior(log_post, theta0, mean, sd, n_chains=256, n_steps=3000, thin=4,
                        seed=0, n_pilot=3, pilot_steps=300, init=1e-4):
    """Preconditioned random-walk Metropolis in standardized coordinates.

    In physical units the five parameters span seventeen orders of magnitude and one proposal
    covariance cannot serve them all. The ensemble starts clustered on theta0 and the proposal
    covariance is re-estimated n_pilot times from the ensemble, growing to fit the posterior;
    acceptance should settle around 0.2-0.4.

    Parameters
    ----------
    log_post : callable
        Maps physical parameters (B, 5) to log posterior densities (B,).
    theta0 : torch.Tensor
        Starting point in physical units, shape (5,).
    mean, sd : torch.Tensor
        Standardization constants, shape (5,).
    n_chains : int, optional
        Chains run in parallel.
    n_steps : int, optional
        Production steps per chain.
    thin : int, optional
        Keep every thin-th production step.
    seed : int, optional
        Seed for the chain randomness.
    n_pilot : int, optional
        Proposal-tuning stages run before production.
    pilot_steps : int, optional
        Steps per pilot stage.
    init : float, optional
        Initial ensemble spread and proposal scale, in standardized units.

    Returns
    -------
    samples : torch.Tensor
        Physical-unit samples, shape (n_chains * n_steps // thin, 5).
    accs : list of float
        Mean acceptance per stage, pilots first and production last.
    """
    torch.manual_seed(seed)
    u = (theta0 - mean) / sd + init * torch.randn(n_chains, PARAM_DIM, device=device)
    lp = log_post(u * sd + mean)
    L = init * torch.eye(PARAM_DIM, device=device)
    accs, keep = [], []
    for stage, ns in enumerate([pilot_steps] * n_pilot + [n_steps]):
        n_acc = 0.0
        for i in range(ns):
            v = u + torch.randn(n_chains, PARAM_DIM, device=device) @ L.T
            lq = log_post(v * sd + mean)
            accept = torch.rand(n_chains, device=device).log() < (lq - lp)
            u = torch.where(accept[:, None], v, u)
            lp = torch.where(accept, lq, lp)
            n_acc += accept.float().mean().item()
            if stage == n_pilot and i % thin == 0:
                keep.append(u.clone())
        accs.append(n_acc / ns)
        if stage < n_pilot:                      # re-tune the proposal from the ensemble
            cov = torch.cov(u.T)
            L = torch.linalg.cholesky(cov + 1e-8 * torch.diag(cov.diag()))
            L = L * 2.38 / np.sqrt(PARAM_DIM)
    return torch.cat(keep) * sd + mean, accs


t0 = time.time()
with torch.no_grad():
    mcmc, accs = reference_posterior(lambda th: log_prior(th) + log_likelihood(th, d_obs),
                                     theta_true[0], THETA_MEAN, THETA_STD)
n_calls = 256 * (3 * 300 + 3000)
print(f"{len(mcmc):,} MCMC samples in {time.time() - t0:.0f} s, "
      f"{n_calls:,} likelihood evaluations")
print(f"acceptance: pilots {', '.join(f'{a:.2f}' for a in accs[:-1])}, "
      f"production {accs[-1]:.2f}")

ratio = (samples.std(0) / mcmc.std(0)).cpu().numpy()
offset = ((samples.mean(0) - mcmc.mean(0)) / mcmc.std(0)).cpu().numpy()
print(f"\n{'':>6} {'sd flow / sd MCMC':>18} {'mean offset [MCMC sd]':>22}")
for n, r, o in zip(names, ratio, offset):
    print(f"{n:>6} {r:>18.2f} {o:>22.2f}")

The two agree on the shape and differ in the details: the flow's $\delta f$ marginal is 20% too wide
and its centre sits about 0.9 MCMC standard deviations low, while the $A$–$\cos\iota$ ridge is 12%
too narrow. That is a useful size of error to see, small enough that the corner plot in §5 looked
fine and large enough to matter. The corner in §6.2 shows it, next to the fix.

The reference is also not a search. The chains were seeded at $\theta_{\rm true}$ and the proposal
was tuned on the target, and it still spent $10^6$ likelihood evaluations; from a cold start, with
the ridge to find first, it is nearer $10^7$. The flow paid its cost once, during training, and
answers in a couple of seconds. With $\sim\!10^4$ resolvable galactic binaries to analyze, each of
them re-analyzed as the data accumulate, that is the trade.

## 6.2 Importance sampling

Running a reference chain per event defeats the purpose. But the flow gives a *density*, not only
samples, allowing for **importance sampling** treating $q$ as proposal. Draw $\theta_i \sim q(\theta \mid d)$ and weight each
draw by
$$
w_i \;=\; \frac{\pi(\theta_i)\,\mathcal{L}(d \mid \theta_i)}{q(\theta_i \mid d)} .
$$
Weighted averages over $\{\theta_i, w_i\}$ are then expectations under the exact posterior, for any
$q$ that covers it. This is where the prior has to be a proper density: a few per cent of the flow's
samples land outside the box, mostly past $\cos\iota = 1$, and they must be given weight zero rather
than the same constant as everything else.

How good $q$ was shows up in the spread of the weights, summarized by the **sample efficiency**
$$
\epsilon \;=\; \frac{n_{\rm eff}}{n} \;=\; \frac{\bigl(\sum_i w_i\bigr)^2}{n \sum_i w_i^2} .
$$
The average of the weights gives the **evidence**
$$
Z = \int \pi\,\mathcal{L}\,d\theta \approx n^{-1}\sum_i w_i.
$$

In [ ]:
with torch.no_grad():
    # log q in physical units: the flow works in standardized theta, so undo that Jacobian.
    u_std = (samples - THETA_MEAN) / THETA_STD
    log_q = flow.log_prob(u_std, d_obs.expand(N_POST, -1)) - torch.log(THETA_STD).sum()
    log_pi = log_prior(samples)
    log_w = log_pi + log_likelihood(samples, d_obs) - log_q

w = (log_w - log_w.max()).exp()          # unnormalized; the overall scale cancels below
eff = (w.sum() ** 2 / (N_POST * (w ** 2).sum())).item()
log_Z = (torch.logsumexp(log_w, 0) - np.log(N_POST)).item()
log_Z_err = np.sqrt((1 / eff - 1) / N_POST)

outside = torch.isneginf(log_pi).float().mean().item()
print(f"outside the prior box: {100 * outside:.2f}% of samples, given weight zero")
print(f"sample efficiency:     eps = {100 * eff:.1f}%  "
      f"({eff * N_POST:.0f} effective samples out of {N_POST:,})")
print(f"log evidence:          {log_Z:.2f} +- {log_Z_err:.2f}  "
      f"(up to the constant dropped in log_likelihood)")

# Reweighted moments, against the MCMC reference.
wn = (w / w.sum())[:, None]
mean_is = (wn * samples).sum(0)
sd_is = ((wn * (samples - mean_is) ** 2).sum(0)).sqrt()
print(f"\n{'':>6} {'sd IS / sd MCMC':>16} {'mean offset [MCMC sd]':>22}")
for n, r, o in zip(names, (sd_is / mcmc.std(0)).cpu().numpy(),
                   ((mean_is - mcmc.mean(0)) / mcmc.std(0)).cpu().numpy()):
    print(f"{n:>6} {r:>16.2f} {o:>22.2f}")

In [ ]:
# Zoom to +-4.5 sigma of the reference posterior, and put all three on the same axes.
lim = [((mcmc[:, i].mean() - 4.5 * mcmc[:, i].std()) * PLOT_SCALE[i],
        (mcmc[:, i].mean() + 4.5 * mcmc[:, i].std()) * PLOT_SCALE[i]) for i in range(PARAM_DIM)]
kw = dict(labels=LABELS, range=[(float(a), float(b)) for a, b in lim], levels=(0.5, 0.9),
          smooth=1.0, plot_datapoints=False, plot_density=False)

fig = corner.corner((samples * PLOT_SCALE).cpu().numpy(), color="C1",
                    hist_kwargs={"density": True}, **kw)
corner.corner((mcmc * PLOT_SCALE).cpu().numpy(), fig=fig, color="k",
              hist_kwargs={"density": True, "ls": "--"},
              contour_kwargs={"linestyles": "dashed"}, **kw)
corner.corner((samples * PLOT_SCALE).cpu().numpy(), fig=fig, color="C0",
              weights=(w / w.sum()).cpu().numpy(), hist_kwargs={"density": True},
              truths=(theta_true * PLOT_SCALE)[0].cpu().numpy(), truth_color="C3", **kw)
fig.suptitle("flow (orange), MCMC reference (black dashed), "
             "flow + importance weights (blue)", y=1.0);

The orange contours (NPE proposal) tend to be slightly wide. This is due to the **mass-covering property** of forward-KL training---which is crucial for IS to work. Indeed, if $q_\phi$ misses a mode during training, it is heavily penalized in the loss. The blue samples are importance-weighted, and they coincide with the black dashed reference. No chain was run, only one likelihood evaluation per sample, and the correction is exact as $n \to \infty$. Nothing that produces samples alone can do this. 

The **sampling efficiency** $\epsilon$ is used as a performance diagnostic. It is the fraction of the drawn samples the weighted set is worth, and it falls as soon as $q$ misses in any direction. At 11% here, most of the loss is
the $\delta f$ offset and the $\delta f$–$\dot f$ correlation the flow got wrong. Unlike the
validation loss it is a statement about *this* observation. The DINGO-IS pipeline uses importance sampling to validate every result and establish credibility
[arXiv:2210.05686](https://arxiv.org/abs/2210.05686).

## 6.3 P-P plots

Both checks so far concern one observation. The **P–P plot** tests the amortized posterior over the
whole prior. Draw an injection $\theta \sim \pi$, simulate data, sample the flow, and record the
quantile of the true value in each one-dimensional marginal. If the posteriors are calibrated those
quantiles are uniform on $[0, 1]$, so each empirical CDF follows the diagonal. The grey bands are the
null distribution of the order statistics, obtained by simulation; the legend gives a
Kolmogorov–Smirnov $p$-value against uniformity.

In [ ]:
def ks_pvalue(q):
    """Asymptotic Kolmogorov-Smirnov p-value against U[0, 1]."""
    n = len(q)
    x = np.sort(q)
    k = np.arange(1, n + 1)
    d = max(np.max(k / n - x), np.max(x - (k - 1) / n))
    lam = (np.sqrt(n) + 0.12 + 0.11 / np.sqrt(n)) * d
    j = np.arange(1, 101)
    return float(np.clip(2 * np.sum((-1) ** (j - 1) * np.exp(-2 * j**2 * lam**2)), 0, 1))


@torch.no_grad()
def truth_quantiles(injections, n_samples=1000, batch=25):
    """Quantile of each true value in the corresponding marginal. (n_inj, 5) -> (n_inj, 5)

    The flow is sampled for a whole batch of observations at once; one injection at a time
    would leave the network almost idle.

    Parameters
    ----------
    injections : torch.Tensor
        True parameters in physical units, shape (n_inj, 5).
    n_samples : int, optional
        Posterior samples drawn per injection.
    batch : int, optional
        Injections pushed through the flow at once.

    Returns
    -------
    numpy.ndarray
        Quantiles in [0, 1], shape (n_inj, 5).
    """
    q = []
    for i in range(0, len(injections), batch):
        th = injections[i:i + batch]
        s = flow.sample(n_samples, simulate(th)) * THETA_STD + THETA_MEAN   # (batch, n, 5)
        q.append((s < th[:, None, :]).float().mean(1))
    return torch.cat(q).cpu().numpy()


def pp_plot(q, labels, ax=None, confidence=(0.68, 0.95, 0.997), n_band=4000, seed=0):
    """Empirical CDF of the quantiles, with null bands and KS p-values.

    Parameters
    ----------
    q : numpy.ndarray
        Quantiles, shape (n_inj, dim).
    labels : list of str
        One legend label per parameter.
    ax : matplotlib.axes.Axes, optional
        Axes to draw on; a new figure is made if None.
    confidence : tuple of float, optional
        Credible levels of the grey null bands.
    n_band : int, optional
        Simulated draws used to build those bands.
    seed : int, optional
        Seed for that simulation.

    Returns
    -------
    matplotlib.axes.Axes
        The axes drawn on.
    """
    n, dim = q.shape
    if ax is None:
        _, ax = plt.subplots(figsize=(5.2, 5.2))
    y = np.linspace(0, 1, n)
    null = np.sort(np.random.default_rng(seed).random((n_band, n)), axis=1)
    for c in confidence:
        lo, hi = np.quantile(null, [(1 - c) / 2, (1 + c) / 2], axis=0)
        ax.fill_betweenx(y, lo, hi, color="k", alpha=0.08, lw=0)
    for j in range(dim):
        ax.plot(np.sort(q[:, j]), y, lw=1.4, label=f"{labels[j]}  ($p$={ks_pvalue(q[:, j]):.2f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("credible level"); ax.set_ylabel("fraction of injections")
    ax.legend(fontsize=8, loc="upper left")
    return ax


N_INJ, N_PP = 400, 1000
torch.manual_seed(0)
t0 = time.time()
q = truth_quantiles(sample_prior(N_INJ), N_PP)
print(f"{N_INJ} injections x {N_PP:,} samples in {time.time() - t0:.0f} s")
print("KS p-values: " + ", ".join(f"{n} {ks_pvalue(q[:, j]):.2f}" for j, n in enumerate(names)))

pp_plot(q, LABELS)
plt.tight_layout()

All five curves stay inside the bands and no $p$-value is small. That is consistent with §6.1, where
the flow was off by $0.9\sigma$ on one observation: an error that is drawn afresh with every noise
realization averages away here. The P–P plot is the weaker test on any single event and the stronger
one on a population, and what it would catch is systematic bias, posteriors too narrow everywhere or
a parameter consistently offset. That is the failure that matters when the same network is applied to
$10^4$ sources, and it is what the workshop challenge is scored on.

---
# 7. Watching the flow transport probability

<!-- *(~3 min)* -->

Fix the observation as context, draw a cloud from $\mathcal{N}(0, I)$, and undo one **complete flow
step** (permutation + `LULinear` + MADE) at a time — the sampling direction. We show the
$A$–$\cos\iota$ plane, colored by where each point *ends up*, so the map reads backwards. Permutations
only *relabel* components, so we follow the relabelling and always plot the two slots that become $A$
and $\cos\iota$.

In [ ]:
IX, IY = 0, 4          # plot A against cos iota
n_show = 3000

flow.eval()
context = d_obs.expand(n_show, -1)
u = torch.randn(n_show, PARAM_DIM, device=device)

steps = list(flow._transform._transforms)     # 5 flow steps, then a trailing linear map

# A permutation only relabels components. Track the relabelling so that every panel plots the
# two slots that end up as theta[IX] and theta[IY]: slots[k] holds those indices in the frame
# just before flow step k, and slots[-1] is the frame of the base distribution.
perms = [m._permutation for m in flow._transform.modules()
         if isinstance(m, transforms.RandomPermutation)]
slots, s = [], torch.arange(PARAM_DIM, device=device)
for p in [None] + perms:
    s = s if p is None else torch.argsort(p)[s]
    slots.append(s)

# Sampling runs the composition backwards, one full step per panel (the trailing linear map
# rides along with the first of them).
groups = [[steps[5], steps[4]]] + [[steps[k]] for k in (3, 2, 1, 0)]
frames = [6, 4, 3, 2, 1, 0]                   # which slot frame each panel lives in
titles = [r"base:  $u \sim \mathcal{N}(0, I)$"] + [
    f"after {k} flow step" + ("s" if k > 1 else "") for k in range(1, 6)]
titles[-1] += "\n" + r"$= \theta$ (standardized)"

snapshots = [u]
with torch.no_grad():
    z = u
    for group in groups:
        for sub in group:
            z, _ = sub.inverse(z, context)
        snapshots.append(z)

color = snapshots[-1][:, IX].cpu().numpy()    # colour by where each point ends up
truth_std = ((theta_true - THETA_MEAN) / THETA_STD)[0].cpu().numpy()

fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.4))
for ax, snap, frame, title in zip(axes.ravel(), snapshots, frames, titles):
    ix, iy = slots[frame][IX], slots[frame][IY]
    ax.scatter(snap[:, ix].cpu(), snap[:, iy].cpu(), s=2, alpha=0.35, c=color, cmap="coolwarm")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(r"slot $\to A$"); ax.set_ylabel(r"slot $\to \cos\iota$")
axes.ravel()[-1].plot(truth_std[IX], truth_std[IY], "*", color="C3", ms=14)
plt.tight_layout()

The isotropic blob becomes the narrow, curved $A$–$\cos\iota$ ridge, the red star marking the truth.
Because the coloring is by final position the map reads backwards: the gradient already visible in
the base panel says which region of $\mathcal{N}(0, I)$ supplies which end of the ridge, and it stays
coherent throughout. That coherence is not incidental, since the map is a diffeomorphism, which is
what makes the density computable.

Watch the axis ranges as much as the shapes. No single affine step could bend a Gaussian this far;
the curvature is what the stacking buys.

---
# 8. Exercises — *work through at home, or here if time remains*

<!-- Each one asks a single question, trains small flows on the budget its heading gives, and puts the
answer below the code; all of them reuse `train`, `create_flow` and the loaders from the main body.
The **challenge** at the end of Part 2a is a different object: an open seven-parameter problem with
no solution provided, scored by the P–P test of §6.3, and costing an afternoon rather than a few
minutes. -->


In [ ]:
EX_EPOCHS_A = 10        # Exercise A: two architectures on one budget
EX_EPOCHS_B = 6         # Exercise B: five configurations, so a shorter budget each

## Exercise A — affine steps or spline steps

<!-- *(trains two flows; about 3 minutes on a laptop CPU)* -->

`create_base_transform` also implements `"rq-coupling"`: a
[rational-quadratic spline](https://arxiv.org/abs/1906.04032) coupling transform. Each parameter is
pushed through a learned monotonic spline instead of an affine map, and the parameters are split in
half rather than ordered, so *both* directions cost one pass.

**Given ten epochs each, which of the two lands closer to the posterior?** Train one flow of each
type, then measure the distance three ways: by validation loss, against the §6.1 reference on the
§5 observation, and by the sample efficiency $\epsilon$ of §6.2.


In [ ]:
results, ex_flows = {}, {}
for base_transform_type in ["maf", "rq-coupling"]:
    torch.manual_seed(0)
    t0 = time.time()
    # YOUR CODE HERE
    ex_flows[base_transform_type] = f
    results[base_transform_type] = (h["val"][-1], time.time() - t0,
                                    sum(p.numel() for p in f.parameters()))

print(f"\n{'transform':>14} {'val loss':>10} {'time [s]':>10} {'params':>12}")
for k, (v, s, p) in results.items():
    print(f"{k:>14} {v:>10.3f} {s:>10.0f} {p:>12,}")

In [ ]:
# Each flow on the section 5 observation, reweighted by the same three lines as 6.2.
N_EX = 20_000
ex_samples, ex_eff, ex_outside = {}, {}, {}
for base_transform_type, f in ex_flows.items():
    f.eval()
    # YOUR CODE HERE


def summarize(label, s, e, o):
    r = (s.std(0) / mcmc.std(0)).cpu().numpy()
    print(f"{label:>20} {100 * e:7.2f}% {100 * o:12.1f}%   " + " ".join(f"{x:6.1f}" for x in r))


print(f"{'':>20} {'eps':>8} {'outside box':>13}   " + " ".join(f"{n:>6}" for n in names))
for base_transform_type in ex_samples:
    summarize(base_transform_type, ex_samples[base_transform_type],
              ex_eff[base_transform_type], ex_outside[base_transform_type])
summarize(f"section 4, {N_EPOCHS} ep", samples, eff, outside)
print("\nthe last five columns are sd(flow) / sd(MCMC), one per parameter; 1.0 is right")

In [ ]:
# The two flows against the reference, on the two planes that carry the shape.
ref = (mcmc * PLOT_SCALE).cpu().numpy()
scaled = {k: (v * PLOT_SCALE).cpu().numpy() for k, v in ex_samples.items()}
truth = (theta_true * PLOT_SCALE)[0].cpu().numpy()
EX_COLOR = {"maf": "C1", "rq-coupling": "C2"}

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8))
ck = dict(levels=(0.5, 0.9), smooth=1.0, plot_datapoints=False, plot_density=False)

for ax, (i, j) in zip(axes, [(1, 2), (0, 4)]):
    xr = tuple(np.quantile(np.concatenate([s[:, i] for s in scaled.values()]), [0.005, 0.995]))
    yr = tuple(np.quantile(np.concatenate([s[:, j] for s in scaled.values()]), [0.005, 0.995]))
    corner.hist2d(ref[:, i], ref[:, j], ax=ax, color="k", range=[xr, yr], fill_contours=True,
                  contourf_kwargs={"colors": ["none", "0.8", "0.55"]},
                  contour_kwargs={"linestyles": "dashed"}, **ck)
    for k, s in scaled.items():
        corner.hist2d(s[:, i], s[:, j], ax=ax, color=EX_COLOR[k], range=[xr, yr],
                      no_fill_contours=True, **ck)
    ax.plot(truth[i], truth[j], "*", color="C3", ms=13)
    ax.set_xlim(*xr); ax.set_ylim(*yr)
    ax.set_xlabel(LABELS[i]); ax.set_ylabel(LABELS[j])

# A flow is supported on all of R^5 and knows nothing of the prior box. fdot is where that
# shows: its posterior is the widest of the five relative to its prior, so a flow that is too
# broad runs off both ends, and 6.2 gives every one of those samples zero weight.
fdot_edges = [PRIOR_LOW[2].item() * PLOT_SCALE[2].item(),
              PRIOR_HIGH[2].item() * PLOT_SCALE[2].item()]
for edge in fdot_edges:
    axes[0].axhline(edge, color="0.35", lw=1.0, ls=":")
axes[0].text(axes[0].get_xlim()[1], fdot_edges[1], r"$\dot f$ prior edges  ",
             fontsize=8, color="0.35", ha="right", va="top")

xr = tuple(np.quantile(np.concatenate([s[:, 1] for s in scaled.values()]), [0.005, 0.995]))
axes[2].hist(ref[:, 1], bins=60, range=xr, density=True, histtype="step", color="k",
             lw=1.5, ls="--", label="MCMC reference")
for k, s in scaled.items():
    axes[2].hist(s[:, 1], bins=60, range=xr, density=True, histtype="step", color=EX_COLOR[k],
                 lw=1.5, label=f"{k}   $\\epsilon$ = {100 * ex_eff[k]:.2f}%")
axes[2].axvline(truth[1], color="C3", lw=1.2)
axes[2].set_xlabel(LABELS[1]); axes[2].set_ylabel("density")
axes[2].legend(fontsize=8, loc="upper left")
plt.tight_layout()

## Exercise B — depth and width

*(trains five flows; about 4 minutes on a laptop CPU)*

**Where does buying capacity stop paying?** Scan depth and width on a six-epoch budget:
`num_flow_steps` in $(2, 5, 10)$ at `hidden_dim = 128`, then `hidden_dim` in $(64, 128, 256)$ at
five steps. Record validation loss, parameter count and wall-clock time for each.


In [ ]:
scan = {}
for num_flow_steps, hidden_dim in [(2, 128), (5, 128), (10, 128), (5, 64), (5, 256)]:
    torch.manual_seed(0)
    # YOUR CODE HERE

print(f"{'steps':>6} {'hidden':>7} {'val loss':>10} {'time [s]':>10} {'params':>12}")
for (ns, hd), (v, s, p) in scan.items():
    print(f"{ns:>6} {hd:>7} {v:>10.3f} {s:>10.0f} {p:>12,}")
print(f"\nrow 2 again, trained for {N_EPOCHS} epochs instead of {EX_EPOCHS_B} "
      f"(section 4): val {history['val'][-1]:.3f}")

## Exercise C — what the on-the-fly noise buys you

*(trains two flows; about 30 seconds on a laptop CPU)*

**What happens to the validation loss if every signal keeps one noise realization for the whole of
training?** Complete `FrozenNoiseDataset` so that it draws the noise once, at construction, then
train the same flow on the same 5000 signals both ways. The training set is small on purpose: 5000
examples against 1.3 million parameters makes the effect unmistakable.


In [ ]:
class FrozenNoiseDataset(Dataset):
    """One noise realization per signal, drawn once and reused."""

    def __init__(self, theta, signals):
        self.theta = theta
        # YOUR CODE HERE

    def __len__(self):
        return len(self.theta)

    def __getitem__(self, idx):
        return self.data[idx], self.theta[idx]


theta_small, signals_small = theta_train[:5000], signals_train[:5000]
histories = {}
for name, dataset_class in [("fresh noise", GBDataset), ("frozen noise", FrozenNoiseDataset)]:
    torch.manual_seed(0)
    # YOUR CODE HERE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, (name, h) in zip(axes, histories.items()):
    ax.plot(h["train"], label="train")
    ax.plot(h["val"], label="validation")
    ax.set_title(name); ax.set_xlabel("epoch")
axes[0].set_ylabel(r"$-\log q(\theta \mid d)$"); axes[0].legend()
plt.tight_layout()

---
# Where this goes next

### The hard part

Section 6.1 put numbers on the "why not just run a sampler" question, and for this problem the
sampler wins: seconds, $10^6$ likelihood evaluations. Amortization pays that cost once rather than
per source, which matters with $\sim\!10^4$ resolvable binaries and matters again every time the
data lengthen and the catalog is refit. But per-source cost is not the real obstacle between this
notebook and LISA.

The obstacle is that the catalog is not given. Some $10^7$ compact binaries emit in the LISA band;
a few $\times 10^4$ are individually resolvable, and the rest blend into a confusion foreground
that is part of the noise. The number of resolvable sources is unknown, neighboring sources
overlap, and the foreground — so the noise model itself — depends on which sources have been
subtracted. Detection, counting and parameter estimation therefore do not separate into $10^4$
copies of this notebook; they form one global fit over a catalog of unknown size. This notebook is
the inner loop of that problem, not its solution. The rest is open, and it is what this week is
for.

### Part 2

- **Wider priors, and evidences.** Part 2a widens the $\delta f$ prior by a factor of 20 and scans it
  in tiles, using the per-tile evidence of §6.2 to decide which tile holds the source. Conditioning
  the data on a frequency proxy is what lets one network cover the whole band.
- **Calibration of the pipeline.** The P–P test of §6.3 is applied to the conditioned pipeline as a
  whole, not to the flow alone, and the workshop challenge is scored the same way.
- **The production code.** DINGO wraps all of it: embedding networks, data conditioning,
  checkpointing, GNPE.

And on **Thursday**, Max Dax solves this same problem with **flow matching** in `dingo.core`: same
simulator, priors and observation, a continuous flow instead of a discrete one.

### References

- Papamakarios, Pavlakou & Murray, [Masked autoregressive flow](https://arxiv.org/abs/1705.07057) (2017)
- Germain et al., [MADE](https://arxiv.org/abs/1502.03509) (2015)
- Durkan et al., [Neural spline flows](https://arxiv.org/abs/1906.04032) (2019)
- Papamakarios et al., [Normalizing flows for probabilistic modeling and inference](https://arxiv.org/abs/1912.02762) (2019)
- Dax et al., [Real-time gravitational-wave science with neural posterior estimation](https://arxiv.org/abs/2106.12594) (2021)
- Dax et al., [Neural importance sampling for rapid and reliable GW inference](https://arxiv.org/abs/2210.05686) (2022)

The flow-visualization panel and training scaffolding are adapted from the IMPRS GW school tutorial,
[nihargupte-ph/imprs-lecture-week](https://github.com/nihargupte-ph/imprs-lecture-week).